# Data generation

Runs the full LLM generation pipeline for each trait and exports `data/v1/prompts.json`.

Requires `OPENROUTER_API_KEY` in `.env` at the project root.

Set `resume: True` (default) to pick up from where a previous run left off without re-spending tokens.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))  # src/notebooks/ -> src/

from lib.data_generation_v1 import Pipeline

In [ ]:
CONFIG = {
    "output_dir": "data/v1",
    "traits": ["politeness", "hedging_confidence"],
    "models": {
        # Generator: needs to be creative and follow complex JSON schemas.
        "generator": {
            "model": "openrouter/openai/gpt-oss-120b",
            "family": "openai",
            "temperature": 0.5,
            "max_output_tokens": 2048,
        },
        # Judge: needs to be critical and reliable, can be a different model/family.
        "judge": {
            "model": "openrouter/openai/gpt-5-nano",
            "family": "openai",
            "temperature": 0.5,
            "max_output_tokens": 1024,
        },
        # Tie-breaker: used when judge score is within the margin — can be the same as judge.
        "tie_breaker_judge": {
            "model": "openrouter/openai/gpt-5-nano",
            "family": "openai",
            "temperature": 0.5,
            "max_output_tokens": 1024,
        },
    },
    "pipeline": {
        "scenarios_per_trait": 50,
        "scenario_batch_size": 5,
        "paraphrases_per_level": 4,
        "min_acceptance_score": 0.75,
        "max_duplicate_jaccard_scenarios": 0.85,
        "max_duplicate_jaccard": 0.85,
        "lexical_baseline_warning_accuracy": 0.65,
        "human_validation_fraction": 0.2,
        "max_workers": 4,
        "resume": True,
    },
    "validation": {
        "warn_if_same_family_generator_and_judge": True,
        "reject_if_same_family_generator_and_judge": False,
    },
}

In [ ]:
pipeline = Pipeline(CONFIG)

In [ ]:
for trait in CONFIG["traits"]:
    print(f"\n{'='*60}\n  {trait}\n{'='*60}")
    pipeline.make_all(trait)


  politeness
resume=true → 50 scenarios already on disk, target=50
Wrote 50/50 scenarios to /Users/francescobraicovich/Documents/Progetti/Bocconi/llm-behaviour-intensity-representation/data/v1/politeness/scenarios.jsonl. Batches: 0/0 ok. Speech-act distribution: {'criticism_or_feedback': 8, 'refusal': 12, 'request': 5, 'apology': 8, 'bad_news_delivery': 7, 'disagreement': 10}
resume=true → 50 ladders already on disk
All ladders already generated
  ! paraphrases failed for politeness-bad-news-delivery-026: TypeError: object of type 'NoneType' has no len()
  ! paraphrases failed for politeness-criticism-or-feedback-005: TypeError: object of type 'NoneType' has no len()
  ! paraphrases failed for politeness-bad-news-delivery-039: TypeError: object of type 'NoneType' has no len()
  ! paraphrases failed for politeness-criticism-or-feedback-043: TypeError: object of type 'NoneType' has no len()
  ! paraphrases failed for politeness-disagreement-017: TypeError: object of type 'NoneType' has 

In [ ]:
pipeline.export_prompts()
print("\nDone. data/v1/prompts.json is ready.")